# Solveit CRAFT UI Architecture

> Map of every UI feature notebook in `nbs/craft/`: what it owns, why it exists, how `craft_dev.build` assembles it into `dist/`, and which Solveit DOM and runtime contracts it relies on.

## Build Pipeline & Load-Order Layers

`craft_dev.build.build_dist()` scans `nbs/craft/*.ipynb` in **sorted filename order** and concatenates exported cells into `dist/`:

- `#| export_ft <file>` evaluates Python `fastcore.xml` `FT` trees at build time and emits compact HTML/SVG strings into `window.__craftTpl` (and config tokens into `window.__craftCfg`).
- `#| export_js <file>` appends raw JavaScript cells to `<file>` with `/* %% <nb> #<cell_id> */` source markers.
- `#| export_js_iife <file> <group>` wraps multiple cells in the same notebook that share `(file, group)` into a single `(() => { ... })();` block, letting large modules be split across narrated cells without losing lexical scope.
- `#| export_css <file>` appends CSS rules to `<file>` in notebook order. `@layer craft-theme, craft-components, craft-mobile;` (declared in `01_theme_and_tokens.ipynb`) scopes `--craft-*` custom-property token blocks, while DOM-styling rules stay unlayered so they never lose to Solveit's unlayered FrankenUI/Tailwind resets; feature-specific `html.craft-mobile` overrides live in their owning notebooks.

Every notebook in `nbs/craft/` must have a unique numeric prefix (`build_dist` enforces this) because prefix order is both JS initialization order and CSS cascade order. `build_dist` validates every `.js` output with `node --check` and runs `tsc --noEmit --allowJs --checkJs` on `dist/CRAFT.js` when `tsc` is installed; `craft-cdp build --sync` copies built assets to the repo root, flags orphaned root `.js`/`.css` files, and rebuilds `chrome-ext/`.

## Feature Map

| Prefix | Notebook | Output(s) | Globals / Responsibilities |
|---|---|---|---|
| `00` | `00_architecture.ipynb` | *(docs only)* | Architecture guide, dependency layers, and runtime contracts |
| `01` | `01_theme_and_tokens.ipynb` | `CRAFT.css` | `@layer craft-theme, craft-components, craft-mobile;` declaration, Tokyo Night Pure palette, `--craft-prose` / `--craft-code` type scale, Encode Sans & Iosevka webfonts, modals, find/replace, `#vars-sb`, task lists, raw cells, ANSI traceback contrast |
| `02` | `02_hotkey_builtins.ipynb` | `hotkey_builtins.js` | Reference snapshot of upstream Solveit keyboard shortcuts |
| `03` | `03_hotkey_manager.ipynb` | `CRAFT.js`, `hotkey_manager.js`, `CRAFT.css` | `window.__hotkeyRegister` bootstrap queue, `window.solveitHotkeys` registry/dispatcher, `.hk-overlay` settings modal |
| `04` | `04_cm_core.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.__craftCMCore` (with JSDoc `@typedef` contracts): `SOLVEIT` DOM/HTMX selector and endpoint registry, CodeMirror 6 loader (local bundle + `esm.sh` fallback), syntax themes, language loaders, `createInlineEditor` (with unified `pointerEdit` guard), `onCells` single-pass cell reconciler, `features` `localStorage` toggle registry, and shared DOM/HTMX/overlay primitives (`el`, `qs`, `qsa`, `on`, `patch`, `store`, `mountTpl`, `mountModal`, `mountMenu`, `mountBar`, `MODAL_OPEN_SEL`, `typingBlocked`, `htmxPost`, `fetchJson`, `onCellHtmx`, `waitForHtmxSettle`, `waitForNewCell`, `onDomChange`, `allCells`, `selectedCells`, `primaryCell`, `cellMtype`, `getCellInputText`, `headingLevel`, `dlgKey`, `sleep`, `notify`, `notifyErr`, `deleteCellEl`) |
| `05` | `05_dialog_chrome.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftMobile` mode toggle, CSS auto-hiding top nav, `window.craftNoMonaco` (disposes Monaco, installs no-op stubs, removes `#full_editor`), `window.craftLauncher` (`Mod+Shift+P` `<dialog>`, including feature toggles), `window.craftA11y` |
| `06` | `06_file_tree.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.solveitFileTree`, `window.craftDotPrompts`: native `<details>`/`<summary>` sidebar tree, inline CRUD, copy/move/delete, uploads, resizable drawer |
| `07` | `07_cell_presentation.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftCellChains`, `window.craftActionMenus`, `window.craftOmit`, `window.craftParts`, pinned/exported/skipped cues, desktop hover/selection outlines |
| `08` | `08_tables.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftTables`: scroll wrapper, numeric right-alignment, click-to-sort flat headers, Copy TSV, light-mode overrides |
| `09` | `09_cell_insertion.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftInsertPreview`, `window.craftDesktopUX` (hover-edge drag-to-insert, fan picker, `Shift+A`/`Shift+B` seam picker, slash-morph, click-margin escape), `window.craftImageDrop` |
| `10` | `10_code_editor_c2.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftC2`: inline CM6 code editor (gutterless view-parity mode), completions, signature help, ghost text, inline AI |
| `11` | `11_markdown_editor.ipynb` | `CRAFT.js`, `craft_md.js`, `CRAFT.css` | `craft_md.js` loader, `window.craftMd`: Obsidian-style Live Preview for note/prompt/response and plain-text raw editor |
| `12` | `12_mobile.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftMobileBar` bottom editing toolbar, edge-bulge insertion, and mobile layout shell (`--craft-*` mobile tokens use `@layer craft-mobile`; feature-specific `html.craft-mobile` DOM rules live in their owning notebooks) |
| `13` | `13_context_control.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftContext`: transactional visibility engine, All-But one-shot scoping, Context Lenses, Saved Presets, Context Budget Meter, What the AI Sees preview, Traceback Quick-Fix |
| `14` | `14_font_settings.ipynb` | `CRAFT.js`, `craft_fonts.js`, `CRAFT.css` | `CRAFT.js` stub (applies saved font vars, lazy-loads `craft_fonts.js` on trigger), `window.craftFonts`: per-cell-type font family/size/line-height/weight modal with live preview, persistence, and CRAFT.css snippet export |
| `15` | `15_structural_ops.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftStruct`: multi-cell batch bar, Merge Cells, Partition Dialog (Split/Fork/Extract), Send to Another Dialog, Alt/Hyper drag & keyboard reordering *(heading section actions hidden — to re-explore: planned for redesign and rebuild)* |
| `16` | `16_navigation_reference.ipynb` | `CRAFT.js`, `CRAFT.css` | `window.craftNavRef`: collapsible Minimap / Outline Rail, Sticky Reference Pane, session-scoped Stale-Output badges, Scratchpad / ephemeral cell flags |
| `17` | `17_cell_history.ipynb` | `CRAFT.js`, `craft_history.js`, `CRAFT.css` | `CRAFT.js` stub (lazy-loads `craft_history.js` on trigger or pending `data-diff`), `window.craftHistory`: session per-cell version snapshots, inline AI edit diff/Accept/Revert bar, and Cell History restore modal |
| `18` | `18_tts_reader.ipynb` | `CRAFT.js`, `craft_tts.js`, `CRAFT.css` | `CRAFT.js` stub (lazy-loads `craft_tts.js` on trigger), `window.craftTTS`: Web Speech API reader with Markdown/code/table cleaning, sentence chunking, active-cell highlighting, and floating player bar |
| `19` | `19_bug_capture.ipynb` | `CRAFT.js`, `craft_bug_capture.js`, `CRAFT.css` | `CRAFT.js` stub (lazy-loads `craft_bug_capture.js` on trigger or saved icon preference), `window.craftBugCapture`: floating bug-report trigger, DOM, console error, metadata, and optional screenshot capture, and submission modal |

## Core Solveit DOM & Lifecycle Constraints

- **HTMX + WebSocket card replacement:** Solveit replaces `.editable` cards in `#dialog-container` when cells run, save, or sync. Per-cell features subscribe via `window.__craftCMCore.onCells(...)` (one batched `MutationObserver` pass per animation frame) and reference Solveit selectors and HTMX endpoints through `window.__craftCMCore.SOLVEIT`.
- **Hot-reload safety and feature toggles:** Feature IIFEs call `window.<global>?.disable?.()` before registering listeners or observers, and toggleable features guard startup with `CORE.features.init(...)` so they can be toggled from `Mod+Shift+P` via `localStorage` without rebuilding (`window.craftNoMonaco` runs unconditionally).
- **Page-local vs server-backed state:** Omit (`craftOmit`) and the third hidden state in `craftParts` are strictly page-local DOM state keyed by stable message IDs; they never mutate dialog JSON or AI-context inclusion.
- **Standalone runtime deployment:** End-user installs via `setup.sh` copy `CRAFT.css` and `CRAFT.js` to `$HOME/` and the 7 sub-assets (`craft_md.js`, `craft_fonts.js`, `craft_history.js`, `craft_tts.js`, `craft_bug_capture.js`, `hotkey_manager.js`, `hotkey_builtins.js`) plus `vscode-icons/` into `$HOME/.craft-assets/`, without requiring Python, `uv`, or Node.